In [258]:
# tool creation

from langchain_core.tools import tool
import requests
from langchain_core.tools import InjectedToolArg
from typing import Annotated

In [259]:
@tool
def get_conversion_factor(base_currency:str,target_currency:str)->float:
    """this function fetches the currency conversion factor between base currency and target currency but does not convert"""
    url = f"https://v6.exchangerate-api.com/v6/ec623dddb623fa3a144e6ea0/pair/{base_currency}/{target_currency}"

    response=requests.get(url)

    return response.json()

In [260]:
get_conversion_factor.invoke({"base_currency":"USD",
                       "target_currency":"INR"})

{'result': 'success',
 'documentation': 'https://www.exchangerate-api.com/docs',
 'terms_of_use': 'https://www.exchangerate-api.com/terms',
 'time_last_update_unix': 1787961601,
 'time_last_update_utc': 'Sat, 29 Aug 2026 00:00:01 +0000',
 'time_next_update_unix': 1788048001,
 'time_next_update_utc': 'Sun, 30 Aug 2026 00:00:01 +0000',
 'base_code': 'USD',
 'target_code': 'INR',
 'conversion_rate': 95.5338}

In [261]:
@tool
def convert_currency(base_currency_value:int,conversion_rate:Annotated[float,InjectedToolArg])->float:
    """this tool convrt one base currency value to another currency value by taking the conversion rate"""

    return base_currency_value*conversion_rate

In [262]:
convert_currency.invoke({"base_currency_value":10,
                "conversion_rate":85.1})

851.0

In [263]:
#tool binding

In [264]:
from langchain_groq import ChatGroq
from langchain_core.messages import HumanMessage
from langchain_huggingface import ChatHuggingFace,HuggingFaceEndpoint
from dotenv import load_dotenv

load_dotenv()

model = HuggingFaceEndpoint(
    repo_id="meta-llama/Llama-3.1-8B-Instruct",
    task="text-generation"
)

llm = ChatHuggingFace(llm=model)

In [265]:
llm_with_tools=llm.bind_tools([get_conversion_factor,convert_currency])

In [ ]:
#tool execution

message = [
    HumanMessage(
        content="""
        I need BOTH operations.
        
        1. Get the INR to USD conversion rate.
        2. convert 9500 INR to USD.
        
        Call both tools in the same response. Do not skip either operation.
        """
    )
]


In [273]:
message

[HumanMessage(content='\n        I need BOTH operations.\n\n        1. Get the INR to USD conversion rate.\n        2. convert 9500 INR to USD.\n\n        Call both tools in the same response. Do not skip either operation.\n        ', additional_kwargs={}, response_metadata={})]

In [274]:
ai_message=llm_with_tools.invoke(message)
message.append(ai_message)

message

[HumanMessage(content='\n        I need BOTH operations.\n\n        1. Get the INR to USD conversion rate.\n        2. convert 9500 INR to USD.\n\n        Call both tools in the same response. Do not skip either operation.\n        ', additional_kwargs={}, response_metadata={}),
 AIMessage(content='', additional_kwargs={'tool_calls': [{'function': {'arguments': '{"base_currency": "INR", "target_currency": "USD"}', 'name': 'get_conversion_factor', 'description': None}, 'id': 'call_tEFyre6tFHihZyfbnRpp4odd', 'type': 'function'}, {'function': {'arguments': '{"base_currency_value": 9500}', 'name': 'convert_currency', 'description': None}, 'id': 'call_7oWf6xVmTdnCi675Gwq9QR9v', 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 43, 'prompt_tokens': 395, 'total_tokens': 438}, 'model_name': 'meta-llama/Llama-3.1-8B-Instruct', 'system_fingerprint': None, 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--01a04bac-6ddb-74d1-91d7-a5a15dc15f42-0', tool_calls

In [275]:
import json

for tool_call in ai_message.tool_calls:
    if tool_call["name"] == "get_conversion_factor":
        tool_message1=get_conversion_factor.invoke(tool_call)

        conversion_rate=json.loads(tool_message1.content)["conversion_rate"]
        message.append(tool_message1)
    if tool_call["name"]=="convert_currency":

        tool_call["args"]["conversion_rate"]=conversion_rate
        tool_message2=convert_currency.invoke(tool_call)
        message.append(tool_message2)

        



In [276]:
message

[HumanMessage(content='\n        I need BOTH operations.\n\n        1. Get the INR to USD conversion rate.\n        2. convert 9500 INR to USD.\n\n        Call both tools in the same response. Do not skip either operation.\n        ', additional_kwargs={}, response_metadata={}),
 AIMessage(content='', additional_kwargs={'tool_calls': [{'function': {'arguments': '{"base_currency": "INR", "target_currency": "USD"}', 'name': 'get_conversion_factor', 'description': None}, 'id': 'call_tEFyre6tFHihZyfbnRpp4odd', 'type': 'function'}, {'function': {'arguments': '{"base_currency_value": 9500}', 'name': 'convert_currency', 'description': None}, 'id': 'call_7oWf6xVmTdnCi675Gwq9QR9v', 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 43, 'prompt_tokens': 395, 'total_tokens': 438}, 'model_name': 'meta-llama/Llama-3.1-8B-Instruct', 'system_fingerprint': None, 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--01a04bac-6ddb-74d1-91d7-a5a15dc15f42-0', tool_calls

In [280]:
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import PromptTemplate

prompt=PromptTemplate(
    template="""you are a simpe currency converter ai chatbot\n
    you have to simply calculate the currency by analysing the entire message\n
    {message}""",
    input_variables=["message"]
)

parser=StrOutputParser()

chain=prompt | llm | parser



In [282]:
result=chain.invoke({"message":message})

In [283]:
result

'To perform the currency conversion:\n\n1. Get the INR to USD conversion rate:\nThe conversion rate is 1 INR = 0.01047 USD\n\n2. Convert 9500 INR to USD:\n9500 INR is equivalent to 99.465 USD'